# *Setup*

In [11]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from sklearn.model_selection import train_test_split

from src.preprocessing import clean_data, get_transformers
from src.evaluation import evaluate_several_models, display_final_comparison_with_highlight
from src.utils import load_models

In [12]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = clean_data(df)

df_X = df.drop(columns='Churn')
df_y = df.loc[:, ['Churn']].copy()

df_X_train, df_X_test, df_y_train, df_y_test = train_test_split(df_X, df_y, test_size=0.25, shuffle=True, stratify=df_y, random_state=7)
print('Dataset de Treinamento:', df_X_train.shape, df_y_train.shape)
print('Dataset de Testes:', df_X_test.shape, df_y_test.shape)

transformers_X, transformer_y = get_transformers(df_X=df_X, df_y=df_y)

models = load_models(['LR', 'DT', 'KNN', 'RF', 'SVM'], path='../models/', format='.pkl')

Dataset de Treinamento: (5282, 19) (5282, 1)
Dataset de Testes: (1761, 19) (1761, 1)


# Comparação Final entre Modelos com Análise de Threshold

## *Threshold* padrão (0.5)
#### Observações da comparação final:
- Com *threshold* padrão, a métrica $\text{F}_{\beta}\text{ -Score}$ ($\beta=2$) é uma forma de levar em conta as diferenças nos custos de escolha das diferentes classes, priorizando a métrica ***recall*** (*churns* identificados) em relação à **precisão** (clientes não impactados desnecessariamente);
- O modelo **KNN** é campeão em *recall* mas tem a menor precisão; apesar disso, apresenta maior equilíbrio entre estas métricas, seja com pesos iguais ou distintos;
- O modelo **LR** apresenta desempenho intermediário em praticamente todas as métricas;
- O modelo **DT** apresenta quase todas as métricas como as piores;
- O modelo **RF** é campeão em precisão mas intermediário nas outras métricas;
- O modelo **SVM** apresenta segundo melhor *recall* e equilíbrio que pende para este.


#### **Conclusão:** 
> O modelo **KNN** deve ser escolhido: Apresenta, principalmente, melhor $\text{F}_2\text{ - Score}$, e também melhores $\text{F}_1\text{ - Score}$ e *recall*.
>
> **Impacto da Escolha:**
> - $\approx 61\%$ de *churns* identificados;
> - $\approx 39\%$ de clientes impactados desnecessariamente;

In [3]:
metrics_final_comparison = evaluate_several_models(models=models,
                                                   transformers_X=transformers_X, transformer_y=transformer_y,
                                                   df_X_train=df_X_train, df_X_test=df_X_test, df_y_train=df_y_train, df_y_test=df_y_test,
                                                   threshold=0.5, beta_fscore=2.)

In [4]:
display_final_comparison_with_highlight(metrics_final_comparison_dict=metrics_final_comparison, beta_fscore=2,
                                        table_title='Métricas no Conjunto de Testes - Comparação Final - Limiar 0.5',
                                        save_table=False, path_table='../reports/tables/final-comparison_standard-threshold.csv')

,LR,DT,KNN,RF,SVM
Acurácia,0.810,0.793,0.794,0.809,0.800
Precisão,0.672,0.677,0.612,0.686,0.637
Recall,0.552,0.418,0.612,0.518,0.567
F1-Score,0.606,0.517,0.612,0.590,0.600
F2.-Score,0.573,0.452,0.612,0.545,0.580
AUROC,0.858,0.837,0.847,0.854,0.847
AUPR,0.679,0.665,0.658,0.683,0.665


###### [[Acesse pelo **NBViewer** para uma visualização completa da tabela]](https://nbviewer.org/github/levyfelipess/projeto_churn-prediction/blob/main/notebooks/03_model-comparison_threshold-tuning.ipynb)

## Ajuste de *Threshold* visando decisões de negócio

#### Considerações:
- A classe de interesse maior, que são os **clientes que cometerão** ***churn***, é representada por **1**;
- A classe dos que **não cometerão** é representada por **0**;
- O custo de errar a classe **1**, ou seja, **classificar um cliente que é propenso a *churn* como não propenso**, é representado por $\text{c}_{10}$; 
- O custo de errar a classe **0**, ou seja, **classificar um cliente que não é propenso a *churn* como propenso**, é representado por $\text{c}_{01}$;
- O custo $\text{c}_{01}$ está relacionado a investir na retenção de clientes que **não irão abandonar o serviço**, alocando recursos desnecessariamente;
- O custo $\text{c}_{10}$ está relacionado a não investir na retenção de clientes **irão abandonar o serviço**, prejudicando a receita recorrente;
#### Consideração final:
> Ambos são prejudiciais, mas no contexto de negócios, **não investir na retenção de clientes que irão praticar** ***churn*** é mais preocupante, portanto **o custo $\text{c}_{10}$ é maior**.


Se um modelo retorna as probabilidades de classificação das classes **1** e **0** como $p_1$ e $p_0=1-p_1$, respectivamente, devemos escolher a classe **1** caso:

$\begin{eqnarray}
&&p_1 \cdot c_{10} > p_0 \cdot c_{01} \Rightarrow \\
&&p_1 \cdot c_{10} > (1-p_1) \cdot c_{01} \Rightarrow \\
&&p_1 > \dfrac{c_{01}}{c_{01}+c_{10}};
\end{eqnarray}$

Se considerarmos que o custo de errar a classe **1** é $R$ vezes o custo de errar a classe **0** ($c_{10}=R \cdot c_{01}$), devemos escolher a classe **1** caso:

$\begin{eqnarray}
&&p_1 > \dfrac{1}{R+1}.
\end{eqnarray}$

### Simulação com $c_{10}$ 2x maior do que $c_{01}$ $(R=2)$
> 
Vamos considerar:
>O **custo de não reter clientes que irão abandonar o serviço** é **2x** maior do que o **custo de investir desnecessariamente em clientes que não o abandonarão**;

Isso significa que devemos escolher a classe **1** caso $p_1 > 1/3$, ou seja, **o novo *threshold* será 0.33.**

#### Observações da comparação final:
- Como as ponderações para os diferentes tipos de predições já estão sendo levadas em conta a partir dos diferentes custos de erros, a ponderação pela métrica $\text{F}_{\beta}\text{ -Score}$ ($\beta=2$) agora tem um impacto menor na escolha final. O impacto maior volta a ser pela métrica $\text{F}_1\text{ -Score}$;
- O modelo **LR** apresentou o melhor equilíbrio entre precisão e *recall*;
- O modelo **DT** compartilha melhor precisão com **LR**, mas pior *recall*, resultando em equilíbrio intermediário;
- O modelo **RF** apresenta desempenho intermediário em praticamente todas as métricas;
- O modelo **KNN** permanece com melhor *recall* e pior precisão, mas agora com o menor equilíbrio;
- O modelo **SVM**, semelhantemente ao **RF**, também apresenta métricas intermediárias, mas equilíbrio levemente melhor do que este.

#### **Conclusão:**
> O modelo **LR** deve ser escolhido: Apresenta principalmente melhor $\text{F}_1\text{ - Score}$, é computacionalmente leve e um dos mais interpretáveis.
>
> **Impacto da Escolha:**
> - $\approx 75\%$ de *churns* identificados;
> - $42\%$ de clientes impactados desnecessariamente;

In [5]:
metrics_final_comparison = evaluate_several_models(models=models,
                                                   transformers_X=transformers_X, transformer_y=transformer_y,
                                                   df_X_train=df_X_train, df_X_test=df_X_test,
                                                   df_y_train=df_y_train, df_y_test=df_y_test,
                                                   threshold=1 / (2 + 1), beta_fscore=2.)

In [6]:
display_final_comparison_with_highlight(metrics_final_comparison_dict=metrics_final_comparison, beta_fscore=2,
                                        table_title='Métricas no Conjunto de Testes - Comparação Final - Limiar 0.33',
                                        save_table=False, path_table='../reports/tables/final-comparison_threshold-tuning_R-2.csv')

,LR,DT,KNN,RF,SVM
Acurácia,0.789,0.786,0.755,0.775,0.772
Precisão,0.580,0.580,0.525,0.557,0.552
Recall,0.747,0.704,0.788,0.739,0.756
F1-Score,0.653,0.636,0.630,0.635,0.638
F2.-Score,0.706,0.676,0.716,0.694,0.704
AUROC,0.858,0.837,0.847,0.854,0.847
AUPR,0.679,0.665,0.658,0.683,0.665


###### [[Acesse pelo **NBViewer** para uma visualização completa da tabela]](https://nbviewer.org/github/levyfelipess/projeto_churn-prediction/blob/main/notebooks/03_model-comparison_threshold-tuning.ipynb)

### Simulação com $c_{10}$ 5x maior do que $c_{01}$ $(R=5)$
Vamos considerar:
>O **custo de não reter clientes que irão abandonar o serviço** é **5x** maior do que o **custo de investir desnecessariamente em clientes que não o abandonarão**;

Isso significa que devemos escolher a classe **1** caso $p_1 > 1/6$, ou seja, **o novo *threshold* será 0.17.**

#### Observações da comparação final:
- Como as ponderações para os diferentes tipos de predições já estão sendo levadas em conta a partir dos diferentes custos de erros, a ponderação pela métrica $\text{F}_{\beta}\text{ -Score}$ ($\beta=2$) agora tem um impacto menor na escolha final. O impacto maior volta a ser pela métrica $\text{F}_1\text{ -Score}$;
- O modelo **LR** apresentou o melhor equilíbrio entre precisão e *recall*;
- O modelo **DT** apresentou praticamente todos os índices como sendo os menores;
- O modelo **RF** apresenta segundo melhor equilíbrio;
- O modelo **KNN** permanece com melhor *recall*, mas agora com o menor equilíbrio;
- O modelo **SVM** apresenta métricas intermediárias.

#### **Conclusão:**
> O modelo **LR** deve ser escolhido: Apresenta principalmente melhor $\text{F}_1\text{ - Score}$, é computacionalmente leve e um dos mais interpretáveis.
>
> **Impacto da Escolha:**
> - $\approx 93\%$ de *churns* identificados;
> - $\approx 54\%$ de clientes impactados desnecessariamente;

In [13]:
metrics_final_comparison = evaluate_several_models(models=models,
                                                   transformers_X=transformers_X, transformer_y=transformer_y,
                                                   df_X_train=df_X_train, df_X_test=df_X_test, df_y_train=df_y_train, df_y_test=df_y_test,
                                                   threshold=1 / (5 + 1), beta_fscore=2.)

In [14]:
display_final_comparison_with_highlight(metrics_final_comparison_dict=metrics_final_comparison, beta_fscore=2,
                                        table_title='Métricas no Conjunto de Testes - Comparação Final - Limiar 0.17',
                                        save_table=False, path_table='../reports/tables/final-comparison_threshold-tuning_R-5.csv')

,LR,DT,KNN,RF,SVM
Acurácia,0.688,0.611,0.641,0.679,0.639
Precisão,0.457,0.400,0.422,0.448,0.419
Recall,0.925,0.936,0.955,0.899,0.936
F1-Score,0.611,0.561,0.585,0.598,0.579
F2.-Score,0.768,0.738,0.762,0.748,0.751
AUROC,0.858,0.837,0.847,0.854,0.847
AUPR,0.679,0.665,0.658,0.683,0.665


###### [[Acesse pelo **NBViewer** para uma visualização completa da tabela]](https://nbviewer.org/github/levyfelipess/projeto_churn-prediction/blob/main/notebooks/03_model-comparison_threshold-tuning.ipynb)

### Simulação com $c_{10}$ 10x maior do que $c_{01}$ $(R=10)$
Vamos considerar:
>O **custo de não reter clientes que irão abandonar o serviço** é **10x** maior do que o **custo de investir desnecessariamente em clientes que não o abandonarão**;

Isso significa que devemos escolher a classe **1** caso $p_1 > 1/11$, ou seja, **o novo *threshold* será 0.09.**

#### Observações da comparação final:
- Como as ponderações para os diferentes tipos de predições já estão sendo levadas em conta a partir dos diferentes custos de erros, a ponderação pela métrica $\text{F}_{\beta}\text{ -Score}$ ($\beta=2$) agora tem um impacto menor na escolha final. O impacto maior volta a ser pela métrica $\text{F}_1\text{ -Score}$;
- O modelo **LR** apresenta o melhor equilíbrio entre precisão e *recall*;
- O modelo **DT** apresenta desempenho intermediário em praticamente todas as métricas;
- O modelo **RF** apresenta segundo melhor desempenho em praticamente todas as métricas;
- O modelo **KNN** apresenta desempenho intermediário em praticamente todas as métricas;
- O modelo **SVM** apresenta melhor *recall*, mas pior precisão e consequentemente pior equilíbrio.

#### **Conclusão:**
> O modelo **LR** deve ser escolhido: Apresenta melhor $\text{F}_1\text{ - Score}$, é computacionalmente leve e um dos mais interpretáveis.
>
> **Impacto da Escolha:**
> - $\approx 96\%$ de *churns* identificados;
> - $\approx 60\%$ de clientes impactados desnecessariamente;

In [15]:
metrics_final_comparison = evaluate_several_models(models=models,
                                                   transformers_X=transformers_X, transformer_y=transformer_y,
                                                   df_X_train=df_X_train, df_X_test=df_X_test, df_y_train=df_y_train, df_y_test=df_y_test,
                                                   threshold=1 / (10 + 1), beta_fscore=2.)

In [16]:
display_final_comparison_with_highlight(metrics_final_comparison_dict=metrics_final_comparison, beta_fscore=2,
                                        table_title='Métricas no Conjunto de Testes - Comparação Final - Limiar 0.09',
                                        save_table=False, path_table='../reports/tables/final-comparison_threshold-tuning_R-10.csv')

,LR,DT,KNN,RF,SVM
Acurácia,0.604,0.573,0.529,0.576,0.476
Precisão,0.398,0.379,0.358,0.381,0.334
Recall,0.959,0.951,0.981,0.961,0.981
F1-Score,0.562,0.541,0.525,0.546,0.498
F2.-Score,0.748,0.730,0.728,0.737,0.707
AUROC,0.858,0.837,0.847,0.854,0.847
AUPR,0.679,0.665,0.658,0.683,0.665


###### [[Acesse pelo **NBViewer** para uma visualização completa da tabela]](https://nbviewer.org/github/levyfelipess/projeto_churn-prediction/blob/main/notebooks/03_model-comparison_threshold-tuning.ipynb)